# GGUF export of the chosen fine-tuned Qwen3-4B (Milestone 4, step 1)

Implements `docs/milestone-2/EVAL_PROTOCOL.md`, Amendment 2, "Export", in one Kaggle session (GPU T4 ×2, internet on, pinned environment):

0. **Conversion environment first (CPU):** the llama.cpp checkout at the pinned commit and a separate venv, created with a pinned `virtualenv` (installed into its own directory; the platform's Python has no working `ensurepip`), with llama.cpp's own conversion requirements. It runs before anything else, so a failure costs minutes, not the merge.
1. **Merge (GPU, evaluation environment):** `python -m packages.hetren_serving.export merge` checks `choice.json` and `checkpoint-0792` by hash and merges the adapter into the fp16 base on **one T4** with `peft merge_and_unload`, as every fp16 evaluation did, with the Milestone 3 libraries.
2. **Conversion (CPU, separate venv):** the venv from step 0 runs `python -m packages.hetren_serving.export convert`: f16 GGUF, then `llama-quantize` to Q8_0 and Q4_K_M (no importance matrix), and the export manifest.
3. **Token parity (CPU, evaluation environment):** for each variant, a CPU-only `llama-server` serves the file and `python -m packages.support_evaluator.gguf_parity` checks the vocabulary and the round trip of every validation prompt and target. Nothing is generated and no case is run or scored.

**Not an evaluation.** No model output is produced; no split is run. Every run of this notebook, including failed ones, is logged in `docs/milestone-4/EXPORT_LOG.md`.

**Outputs** (`/kaggle/working/gguf-export/`, attachable by the Kaggle CPU pilot): the Q8_0 and Q4_K_M files, `export.manifest.json`, `merge.json`, `tokenizer/` (the merged model's non-weight files, for the parity driver and the pilot), `parity-<variant>.json`, `environment.json` and `outputs.sha256`. The merged weights and the f16 GGUF stay in scratch space: they are recorded by hash in the manifest but not kept, to stay within Kaggle's output limit.

**Attach:** the evaluation bundle (built by `scripts/build_kaggle_bundle.py` at the commit set below), session 2's training output (holding `checkpoint-0792`) and the checkpoint-selection output (holding `choice.json`).

In [ ]:
BUNDLE_COMMIT = ""  # the bundle's commit (full sha); the notebook stops if the bundle found is another
STATE_SHA256 = "d6230ff6e1780ac5286f0accc0a45298dd8db49fc11ed6fdb1330d98189276d5"  # checkpoint-0792 (validation log)
CHOICE_SHA256 = "d41a7ef595e8882ff0dbb367e62b03b87c8f1006bb60da09811fb19f11992332"  # selection-run-1 choice.json
LLAMA_CPP_COMMIT = "620d2225cc436ab0b001e3200e590e3b286ef9ab"  # release v0.7.0 (Amendment 2)
VARIANTS = ["Q8_0", "Q4_K_M"]
VIRTUALENV = "21.14.13"  # creates the conversion venv with its own bundled pip (no ensurepip needed)
SCRATCH_GB = 40  # merged weights, f16 GGUF, both variants, the base model cache and the venv, with a margin

import glob, hashlib, json, os, platform, shutil, subprocess, sys, tarfile, time, urllib.request

INPUTS, WORK, SCRATCH, OUT = "/kaggle/input", "/kaggle/working/hetren", "/tmp/hetren-export", "/kaggle/working/gguf-export"


def earlier_session(path):  # a copy of the code inside an attached earlier notebook output, not the bundle itself
    return bool(glob.glob(f"{path.split('/hetren/')[0]}/qlora-*")) if "/hetren/" in path else False


if not os.path.exists(WORK):
    unpacked = [p for p in glob.glob(f"{INPUTS}/**/BUNDLE.json", recursive=True) if not earlier_session(p)]
    packed = [p for p in glob.glob(f"{INPUTS}/**/hetren-*bundle.tar.gz", recursive=True) if not earlier_session(p)]

    if len(unpacked) > 1 or (not unpacked and len(packed) != 1):
        raise SystemExit(f"STOP: attach exactly one bundle; found {unpacked or packed}")

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

    print("bundle from", unpacked[0] if unpacked else packed[0])

COMMIT = json.load(open(f"{WORK}/BUNDLE.json"))["commit"]
print("bundle commit:", COMMIT)

if not BUNDLE_COMMIT or COMMIT != BUNDLE_COMMIT.strip():
    raise SystemExit(f"STOP: the bundle is commit {COMMIT}, not BUNDLE_COMMIT {BUNDLE_COMMIT!r}")

if os.path.exists(OUT) and os.listdir(OUT):
    raise SystemExit(f"STOP: {OUT} is not empty; this notebook exports once per session")

os.makedirs(SCRATCH, exist_ok=True)
free = shutil.disk_usage(SCRATCH).free / 1e9
print(f"free scratch space: {free:.1f} GB")

if free < SCRATCH_GB:
    raise SystemExit(f"STOP: {free:.1f} GB free in {SCRATCH}; at least {SCRATCH_GB} GB are needed")

os.chdir(WORK)

## 0. llama.cpp checkout and the conversion venv (before the merge)

The venv is created by `virtualenv` installed into its own directory (`venv-tools`), so neither tool nor converter packages enter the evaluation environment. It has no access to the system site-packages.

In [ ]:
LLAMA = f"{SCRATCH}/llama.cpp"

if not os.path.exists(LLAMA):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)

subprocess.run(["git", "-C", LLAMA, "checkout", "--quiet", LLAMA_CPP_COMMIT], check=True)
head = subprocess.run(["git", "-C", LLAMA, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()

if head != LLAMA_CPP_COMMIT:
    raise SystemExit(f"STOP: llama.cpp is at {head}, not {LLAMA_CPP_COMMIT}")

TOOLS, VENV = f"{SCRATCH}/venv-tools", f"{SCRATCH}/convert-venv"
VPY = f"{VENV}/bin/python"
started = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", TOOLS, f"virtualenv=={VIRTUALENV}"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--quiet", "--no-periodic-update", VENV],
               env={**os.environ, "PYTHONPATH": TOOLS}, check=True)
subprocess.run([VPY, "-m", "pip", "--version"], check=True)
prefix, base = json.loads(subprocess.run([VPY, "-c", "import json, sys; print(json.dumps([sys.prefix, sys.base_prefix]))"],
                                          capture_output=True, text=True, check=True).stdout)

if os.path.realpath(prefix) != os.path.realpath(VENV) or os.path.realpath(base) == os.path.realpath(prefix):
    raise SystemExit(f"STOP: {VPY} is not an isolated venv at {VENV}: prefix {prefix}, base {base}")

print(f"conversion venv created in {time.time() - started:.0f} s")
subprocess.run([VPY, "-m", "pip", "install", "-q", "-r", f"{LLAMA}/requirements/requirements-convert_hf_to_gguf.txt"], check=True)
subprocess.run([VPY, "-m", "pip", "install", "-q", f"{LLAMA}/gguf-py"], check=True)  # the checkout's gguf, as the converter uses
converter = subprocess.run([VPY, "-c", "import gguf, torch, transformers; print(transformers.__version__, torch.__version__)"],
                           capture_output=True, text=True, check=True).stdout.strip()
print(f"conversion venv ready in {(time.time() - started) / 60:.1f} min: transformers, torch = {converter}")

## 1. Merge on one T4 (evaluation environment, Milestone 3 libraries)

In [ ]:
# The evaluation libraries, as in the Milestone 3 runs; torch is the platform's.
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1 peft==0.17.1
GPUS = [line.split(",")[1].strip() for line in subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"], capture_output=True, text=True).stdout.splitlines()]

if not GPUS or "T4" not in GPUS[0]:
    raise SystemExit(f"STOP: the GPUs are {GPUS}; the merge runs on a T4")


def find(pattern, digest):
    found = sorted(glob.glob(f"{INPUTS}/**/{pattern}", recursive=True))
    matching = [p for p in found if hashlib.sha256(open(p, "rb").read()).hexdigest() == digest]

    if not matching:
        raise SystemExit(f"STOP: no {pattern} with sha256 {digest} among {found}")

    return matching[0]


CHECKPOINT = os.path.dirname(find("checkpoint-0792/state.json", STATE_SHA256))
CHOICE = find("qlora-selection/choice.json", CHOICE_SHA256)
print("checkpoint:", CHECKPOINT, "\nchoice:", CHOICE)
MERGED = f"{SCRATCH}/merged"
started = time.time()
subprocess.run([sys.executable, "-m", "packages.hetren_serving.export", "merge", "--checkpoint", CHECKPOINT,
                "--choice", CHOICE, "--out", MERGED, "--device", "cuda"],
               env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}, check=True)
print(f"merged in {(time.time() - started) / 60:.1f} min")

## 2. Build llama.cpp at the pinned commit (CPU only)

In [ ]:
CMAKE_FLAGS = ["-DCMAKE_BUILD_TYPE=Release", "-DGGML_CUDA=OFF", "-DGGML_METAL=OFF", "-DGGML_BLAS=OFF", "-DGGML_NATIVE=OFF"]
started = time.time()
subprocess.run(["cmake", "-S", LLAMA, "-B", f"{LLAMA}/build", *CMAKE_FLAGS], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["cmake", "--build", f"{LLAMA}/build", "--target", "llama-quantize", "llama-server", "-j", str(os.cpu_count())],
               check=True, stdout=subprocess.DEVNULL)
print(f"built llama-quantize and llama-server in {(time.time() - started) / 60:.1f} min")

## 3. Convert to f16 and quantise (CPU, conversion venv)

In [ ]:
GGUF = f"{SCRATCH}/gguf"
os.makedirs(OUT, exist_ok=True)
started = time.time()
subprocess.run([VPY, "-m", "packages.hetren_serving.export", "convert", "--merged", MERGED, "--llama-cpp", LLAMA,
                "--out", GGUF, "--manifest", f"{OUT}/export.manifest.json"], check=True)
print(f"converted and quantised in {(time.time() - started) / 60:.1f} min")

manifest = json.load(open(f"{OUT}/export.manifest.json"))
names = {variant: f"hetren-qwen3-4b-ckpt0792-{variant.lower()}.gguf" for variant in VARIANTS}

for variant, name in names.items():
    shutil.copy2(f"{GGUF}/{name}", f"{OUT}/{name}")

shutil.copy2(f"{MERGED}/merge.json", f"{OUT}/merge.json")
shutil.copytree(MERGED, f"{OUT}/tokenizer", ignore=shutil.ignore_patterns("*.safetensors", "*.safetensors.index.json", "merge.json"))
print(json.dumps(manifest["gguf"], indent=2))

## 4. Token parity for each variant (CPU `llama-server`, evaluation environment)

`gguf-py` from the pinned checkout is installed into the evaluation environment without dependencies, so no installed library changes.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", f"{LLAMA}/gguf-py"], check=True)
SERVER = f"{LLAMA}/build/bin/llama-server"
PARITY = {}

for variant, name in names.items():
    server = subprocess.Popen([SERVER, "-m", f"{OUT}/{name}", "--host", "127.0.0.1", "--port", "8080", "-c", "4096",
                               "-np", "1", "-ngl", "0", "-t", str(os.cpu_count())],
                              stdout=open(f"{SCRATCH}/server-{variant}.log", "w"), stderr=subprocess.STDOUT)

    try:
        for _ in range(300):
            try:
                if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=5).status == 200:
                    break
            except Exception:
                time.sleep(1)
        else:
            raise SystemExit(f"STOP: llama-server did not become ready for {variant}; see {SCRATCH}/server-{variant}.log")

        started = time.time()
        PARITY[variant] = subprocess.run([sys.executable, "-m", "packages.support_evaluator.gguf_parity",
                                          "--merged", f"{OUT}/tokenizer", "--gguf", f"{OUT}/{name}",
                                          "--server", "http://127.0.0.1:8080", "--out", f"{OUT}/parity-{variant}.json"]).returncode
        print(f"{variant}: parity exit {PARITY[variant]} in {(time.time() - started) / 60:.1f} min")
    finally:
        server.terminate()
        server.wait(timeout=60)

## Outputs

In [ ]:
def versions(python):
    return subprocess.run([python, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.splitlines()


environment = {
    "bundle_commit": COMMIT, "llama_cpp_commit": head, "cmake_flags": CMAKE_FLAGS,
    "platform": platform.platform(), "cpu": subprocess.run(["lscpu"], capture_output=True, text=True).stdout,
    "gpu_used_for_merge": GPUS[0], "evaluation_environment": versions(sys.executable),
    "conversion_venv": versions(VPY), "venv_tools": f"virtualenv=={VIRTUALENV}",
    "venv_tools_packages": subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze", "--path", TOOLS],
                                          capture_output=True, text=True).stdout.splitlines(),
    "parity_exit": PARITY,
}
json.dump(environment, open(f"{OUT}/environment.json", "w"), indent=2)

with open(f"{OUT}/outputs.sha256", "w") as listing:
    for path in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
        if os.path.isfile(path) and not path.endswith("outputs.sha256"):
            digest = hashlib.sha256()

            with open(path, "rb") as handle:
                for chunk in iter(lambda: handle.read(1 << 20), b""):
                    digest.update(chunk)

            listing.write(f"{digest.hexdigest()}  {os.path.relpath(path, OUT)}\n")

print(open(f"{OUT}/outputs.sha256").read())

if any(code != 0 for code in PARITY.values()):
    raise SystemExit(f"token parity failed: {PARITY}; see parity-<variant>.json. Log the run before anything else")